In [ ]:
# revisemos la data de  merged/causal_model_data.csv
import pandas as pd
df = pd.read_csv("data/clean/merged/causal_model_data.csv")
df

## 1. Filtro de ventana temporal y validación de cobertura

El archivo `causal_model_data.csv` ya viene generado con ventana 2021-2025 desde `00_prep_dataset.ipynb`. Este filtro es una capa de seguridad (defensivo), no una limpieza necesaria hoy — protege contra el día en que `00_prep_dataset` se vuelva a correr con datos más recientes (SIAF y SIEN en crudo ya tienen 2026).

In [ ]:
df = df[df["anio"].between(2021, 2025)].copy()

print(f"Shape tras filtro de años: {df.shape}")
print(f"Años presentes: {sorted(df['anio'].unique())}")
print(f"Distritos únicos: {df['ubigeo'].nunique()}  (debería ser 1,889 — el universo completo)")


## 2. Revisión de nulos

X (contexto territorial) no debería tener ningún nulo — ya se validó en `00_prep_dataset.ipynb`. Los únicos nulos esperados son en las columnas de Y y T, exactamente en las mismas 146 filas (`y_no_disponible` y `t_no_disponible` coinciden siempre, porque `gasto_total_percapita` se calcula dividiendo entre `ninos_evaluados`, no entre población — cuando no hubo niños tamizados ese año, se cae tanto el numerador de Y como el denominador de T).

In [ ]:
nulos = df.isnull().sum()
print("Columnas con nulos:")
print(nulos[nulos > 0])

print()
print(f"y_no_disponible = 1: {df['y_no_disponible'].sum()} filas")
print(f"t_no_disponible = 1: {df['t_no_disponible'].sum()} filas")
print(f"Coinciden exactamente: {((df['y_no_disponible']==1) == (df['t_no_disponible']==1)).all()}")


## 3. Dataset para el modelo Y ~ X ("sin gasto")

Este notebook entrena **solo** el modelo auxiliar de DML que predice `prevalencia_anemia` a partir de X (contexto territorial), sin ver nunca el gasto (T). El residuo de este modelo (Y real - Y predicho) es el insumo que después usará el causal forest.

**Filtrado:** se excluyen las 146 filas con `y_no_disponible = 1` — no se imputa el resultado, simplemente no hay target real contra qué entrenar en esas filas. Esto no borra esas filas del archivo maestro (`causal_model_data.csv` las conserva con su bandera), solo las saca de este entrenamiento puntual.

**`muestra_pequena` (950 filas, Y ruidoso por `ninos_evaluados` < 10):** queda pendiente de decisión — por ahora se conserva en el dataset de entrenamiento sin ponderar ni excluir.

In [ ]:
cols_X = [
    "pct_cultivo", "pct_construido", "pct_desnudo", "pct_agua_visible",
    "n_edificios", "area_construida_m2", "confianza_media",
    "area_distrito_km2", "densidad_edificios_km2",
    "elevacion_media", "pendiente_media",
    "pct_agua_permanente", "pct_agua_estacional",
    "altitude", "superficie", "pob_densidad_2020",
]

anemia_model_df = df[df["y_no_disponible"] == 0].copy()

print(f"Filas excluidas (sin dato de anemia): {(df['y_no_disponible']==1).sum()}")
print(f"Filas para entrenar Y~X: {anemia_model_df.shape[0]}")
print(f"Distritos únicos en el subset de entrenamiento: {anemia_model_df['ubigeo'].nunique()}")
print(f"  -> de los {df['ubigeo'].nunique()} distritos totales, "
      f"{df['ubigeo'].nunique() - anemia_model_df['ubigeo'].nunique()} quedan sin NINGUNA fila con Y disponible")


In [ ]:
X = anemia_model_df[cols_X]
y = anemia_model_df["prevalencia_anemia"]

assert X.isnull().sum().sum() == 0, "X tiene nulos, revisar"
assert y.isnull().sum() == 0, "y tiene nulos, revisar"
assert anemia_model_df["anio"].between(2021, 2025).all(), "Hay años fuera de 2021-2025"

print("X e y sin nulos, ventana 2021-2025 confirmada ✅")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")


## 4. Modelo: LightGBM, y por qué

Se usa **LightGBM** (gradient boosting sobre árboles) para predecir `prevalencia_anemia` a partir de X. Razones concretas para esta tabla (9,299 filas, 16 columnas numéricas):

- Captura relaciones no lineales e interacciones entre variables territoriales (ej. altitud y dispersión pueden interactuar) sin que haya que especificarlas a mano, a diferencia de una regresión lineal.
- Es rápido de entrenar y de afinar con muchas corridas de validación cruzada — importante porque el paso siguiente (Optuna) entrena cientos de modelos.
- Es la elección por defecto más común como `model_y` / `model_t` dentro de EconML/`CausalForestDML`, así que este mismo modelo (con estos mismos hiperparámetros) es directamente reusable cuando armes el causal forest.

In [ ]:
import lightgbm as lgb
import optuna
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
from pathlib import Path

optuna.logging.set_verbosity(optuna.logging.WARNING)

# el grupo de la validación cruzada es el DISTRITO (ubigeo), no la fila
groups = anemia_model_df["ubigeo"]
N_SPLITS = 5


## 5. Ajuste de hiperparámetros con Optuna

**Por qué agrupar por `ubigeo` en vez de un K-Fold normal:** el panel tiene el mismo distrito repetido hasta 5 veces (2021-2025), y varias columnas de X (`altitude`, `superficie`, `pob_densidad_2020`, etc.) casi no cambian de un año a otro para un mismo distrito. Con un K-Fold normal, el modelo podría ver a Ayacucho-2022 en entrenamiento y a Ayacucho-2023 en validación — casi la misma fila — y parecer mejor de lo que realmente es. `GroupKFold` por `ubigeo` garantiza que un distrito completo (sus 5 años) caiga siempre del mismo lado.

`n_trials = 80`: se sube de lo mínimo (30) porque este notebook no se va a volver a tunear — se busca el mejor modelo posible en una sola corrida, aunque tome varios minutos. Se puede dejar corriendo y volver después.

In [ ]:
def objective(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 7, 63),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 60),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    gkf = GroupKFold(n_splits=N_SPLITS)
    rmses = []
    for train_idx, val_idx in gkf.split(X, y, groups):
        model = lgb.LGBMRegressor(**params, random_state=42)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = model.predict(X.iloc[val_idx])
        rmses.append(mean_squared_error(y.iloc[val_idx], pred) ** 0.5)
    return float(np.mean(rmses))

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=80, show_progress_bar=True)

best_params = study.best_params
print(f"Mejor RMSE (CV): {study.best_value:.4f}")
print(f"Mejores hiperparámetros: {best_params}")


## 6. Cross-fitting honesto: predicción out-of-fold y evaluación

Con los mejores hiperparámetros, se repite el mismo esquema de 5 folds por distrito, pero ahora para quedarnos con la predicción de cada fila **hecha por un modelo que nunca vio ese distrito** — eso es lo que la convierte en una predicción honesta, no sobreajustada. Cada fila termina con `pred_anemia_oof` y su residuo `residuo_Y = prevalencia_anemia - pred_anemia_oof`, el insumo que necesita el causal forest.

In [ ]:
gkf = GroupKFold(n_splits=N_SPLITS)
oof_pred = np.zeros(len(X))
fold_r2, fold_rmse = [], []
feature_importances = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    model = lgb.LGBMRegressor(**best_params, random_state=42, verbosity=-1)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict(X.iloc[val_idx])
    oof_pred[val_idx] = pred

    r2 = r2_score(y.iloc[val_idx], pred)
    rmse = mean_squared_error(y.iloc[val_idx], pred) ** 0.5
    fold_r2.append(r2); fold_rmse.append(rmse)
    feature_importances.append(model.feature_importances_)
    print(f"Fold {fold}: R² = {r2:.3f}  RMSE = {rmse:.4f}  (n_val={len(val_idx)})")

print()
print(f"R² promedio (CV, por distrito nunca visto): {np.mean(fold_r2):.3f} ± {np.std(fold_r2):.3f}")
print(f"RMSE promedio (CV): {np.mean(fold_rmse):.4f}")
print(f"R² global sobre todas las predicciones out-of-fold: {r2_score(y, oof_pred):.3f}")


**Sobre el R² esperado:** no va a salir un R² alto (típicamente ronda 0.10-0.15), y eso es normal, no un error — la anemia infantil depende de muchos factores que no son territoriales (nutrición del hogar, lactancia, anemia materna, calidad de la atención de salud), y este modelo solo ve el contexto que capta el satélite. Para DML **no hace falta que este modelo sea muy preciso** — solo que sea honesto (out-of-fold) y capture la señal real que sí existe en X, para poder restársela a Y antes de estimar el efecto del gasto.

## 7. Importancia de variables — explicando qué está usando el modelo

Responde "qué variable de contexto territorial pesa más para predecir anemia" — sirve de chequeo de sensatez y como insumo para explicar el modelo en la presentación.

In [ ]:
importancia = pd.DataFrame(feature_importances, columns=cols_X).mean().sort_values(ascending=False)

print("Importancia promedio de cada variable de X (gain, promedio entre los 5 folds):")
print(importancia)

importancia.plot(kind="barh", figsize=(8, 6), title="Importancia de variables — Y ~ X (anemia)")


## 8. Guardar resultados en `data/predictions/`

Se agregan `pred_anemia_oof` y `residuo_Y` a `anemia_model_df` y se exporta a `data/predictions/` (no a `clean/merged`, que es solo para paneles de entrada) — para que el notebook del causal forest los lea directo sin reentrenar este modelo.

In [ ]:
anemia_model_df["pred_anemia_oof"] = oof_pred
anemia_model_df["residuo_Y"] = anemia_model_df["prevalencia_anemia"] - anemia_model_df["pred_anemia_oof"]

OUTPUT_DIR = Path("data/predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cols_salida = ["ubigeo", "anio", "prevalencia_anemia", "pred_anemia_oof", "residuo_Y"]
anemia_model_df[cols_salida].to_csv(OUTPUT_DIR / "residuos_anemia_model.csv", index=False)

print(f"Guardado: {OUTPUT_DIR / 'residuos_anemia_model.csv'}  ({len(anemia_model_df)} filas)")
anemia_model_df[cols_salida].head()
